# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [14]:
!{sys.executable} scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queu

**Lane: Refresh / Content Opportunity Scoring.** FlyRank's review team has limited capacity
and a large, aging content inventory — the question isn't "is this page perfect," it's "which
of thousands of pages should a human open first." That's a ranking problem: score every page,
order by evidence of decline plus visible demand, and hand reviewers a short, explainable queue
instead of a spreadsheet. I already have working pieces of this from the starter notebooks (a
hand rule, a readable decision tree, honest client-holdout evaluation), so this lane lets me
build toward a real capstone instead of starting a new problem from zero.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
LANE = "Refresh / Content Opportunity Scoring"
FREESTYLE = False
print(f"Lane selected: {LANE}")
print(f"Freestyle: {FREESTYLE}")





Lane selected: Refresh / Content Opportunity Scoring
Freestyle: False


**Unit of analysis:** one row = one content page for one client (`content_id` within
`client_id`) — not a client, not a day.

**Decision improved:** which pages a content reviewer opens *first* out of a much larger set
they can't all check by hand each week.

**Who acts:** the content/SEO review team, working against a fixed weekly review capacity —
not every flagged page gets touched.

**Action taken:** based on reason codes, a reviewer refreshes the content, reviews CTR/metadata,
checks engagement, expands a thin page, or leaves it on monitor.

**Cost of a wrong call:** a false positive wastes a reviewer's limited time on a page that
wasn't actually a problem — capacity that could have gone to a real decline. A false negative
is quieter and worse: a genuinely declining page never surfaces, so it keeps losing visibility
uncorrected until someone notices by chance. Because review capacity is scarce, ranking quality
(who's in the top 20–50) matters more than raw accuracy across all 30,000 pages.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [16]:

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

n_pages = df["content_id"].nunique()
n_clients = df["client_id"].nunique()
dup_pages = df["content_id"].duplicated().sum()

print(f"Distinct pages (content_id): {n_pages}")
print(f"Distinct clients: {n_clients}")
print(f"Duplicate content_id rows: {dup_pages}  <- should be 0, confirms one row = one page")




Distinct pages (content_id): 30000
Distinct clients: 32
Duplicate content_id rows: 0  <- should be 0, confirms one row = one page


Over half the eligible content (54.2%) is trending down, so this isn't a rare-event problem —
it's a volume problem the review team can't brute-force. A tight, obvious hand rule (stale
*and* visible) is almost always right when it fires (94.1% precision) but barely finds any of
the problem — it flags just 17 of 30,000 pages and catches 0.10% of all declining pages. That
gap between "high-precision but nearly blind" and "the actual scale of decline" is the
opportunity: a ranking approach that can also use softer signals, like the moderate
avg_position–CTR relationship (-0.239), is what a hand rule alone can't do.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

elig = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")

n_total = len(elig)
n_declining = (elig["trend_direction"] == "down").sum()
print(f"Eligible pages: {n_total}")
print(f"Declining-labeled pages: {n_declining} ({n_declining/n_total:.1%})")

stale_visible = elig[(elig["days_since_last_update"] >= 180) & (elig["impressions_90d"] >= 500)]
caught = (stale_visible["trend_direction"] == "down").sum()
print(f"\n'Stale + visible' hand rule flags: {len(stale_visible)} pages ({len(stale_visible)/n_total:.1%} of eligible)")
print(f"  Precision: {caught/len(stale_visible):.1%}  |  Recall: {caught/n_declining:.2%}")

visible = elig[elig["impressions_90d"] >= 100]
corr = visible["avg_position"].corr(visible["ctr"])
print(f"\nCorrelation avg_position vs ctr (visible pages): {corr:.3f}")

Eligible pages: 30000
Declining-labeled pages: 16262 (54.2%)

'Stale + visible' hand rule flags: 17 pages (0.1% of eligible)
  Precision: 94.1%  |  Recall: 0.10%

Correlation avg_position vs ctr (visible pages): -0.239


**What this work can say:** which observed signals are *associated* with decline; how a simple
rule and a model *rank* pages relative to each other; a ranked, explainable review queue with
reason codes; a decision-support tool that shrinks the pages a human has to check first.

**What it can never say:** that refreshing a page *causes* recovery (that needs an actual
experiment, not this data); anything about Google's ranking algorithm; anything framed as
guaranteed rather than observed/directional. The current label (`trend_direction == "down"`)
is also a *proxy* — calculated from the current window, not a true future outcome — so early
claims stay modest until it becomes a real prior-window → future-window label. No client names,
domains, or URLs appear anywhere in this notebook — only pseudonymized IDs and aggregates.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [18]:

forbidden_terms = ["url", "domain", "query", "title", "client_name", "keyword_text"]
flagged_cols = [c for c in df.columns if any(term in c.lower() for term in forbidden_terms)]
print("Columns matching forbidden raw-identifier patterns:", flagged_cols if flagged_cols else "None found")
print(f"\nTotal columns in starter dataset: {len(df.columns)}")

Columns matching forbidden raw-identifier patterns: None found

Total columns in starter dataset: 44


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.